In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.plot  # submodule is not imported by `import rasterio` alone
import torch
import time

import networkx as nx
import base64
from IPython.display import Image  # For displaying the training GIF - doesn't work in Quarto, but does in Jupyter
from IPython.display import HTML
from IPython.display import IFrame
from rasterio.transform import Affine
from matplotlib.colors import LogNorm

# Use this to load the functions used in this notebook
# You can also 'import deepssf' and call functions as deepssf.function_name
import deepssf

print(f"deepssf {deepssf.__version__}")
print(f"torch {torch.__version__}")

# Setup the script

## Set random seed and the device for predicting from the model

If you have a GPU, this is where it will be specified (automatically).

In [ ]:
# Reproducibility — fix random seeds so results are the same on every run.
# Prediction is not deterministic: simulate_trajectory() samples a next location
# from the predicted probability surface at every step, so without a seed two
# runs of the same trained model give different trajectories (and a different
# heatmap).  Seeding here makes a set of simulations reproducible.
torch.manual_seed(42)
np.random.seed(42)

# DEVICE is capitalised — module-level constant (set once, never reassigned).
# get_device() returns 'cuda' if a GPU is available, 'mps' on Apple Silicon,
# or 'cpu' otherwise.  All tensors and the model are moved to this device.
DEVICE = deepssf.get_device()
print(f"Device: {DEVICE}")

## Set script paths
These must point at the same data and rasters the model was trained on — the
model takes one input channel per raster band, in `LAYER_PATHS` order, so a
different set of layers (or a different order) either fails to load the weights
or feeds the wrong covariate into each filter.

`OUTPUT_DIR` is the directory a training run wrote its checkpoints to, given
here as a literal path rather than through `get_output_dir()` (which mints a
*new* dated directory).  Figures and rasters produced below are written back
into the same directory, alongside the model they came from.

*All-caps names below are module-level constants — set once here and referenced throughout the notebook without being reassigned.  Python has no built-in constant keyword, so capitalisation is the community convention (PEP 8) to signal "do not change this value".*


In [ ]:
DATA_DIR = Path("data")  # location of the bundled sample data
MAPPING_DIR = Path("mapping")  # location of the landcover mapping files
CSV_PATH = DATA_DIR / "pigs_east_alligator_steps_covs_20261015.csv"  # GPS tracking data, 28 individuals

# Must match LAYER_PATHS in deepssf_piggies_train_landcover.ipynb, in the same order.
# NOTE: 's2' and 's2_dir' are RESERVED keys in load_environmental_layers().
# 's2_dir' triggers the monthly-composite loader (load_s2_data), and a plain
# 's2' entry is skipped by the per-file loop — so a single multi-band GeoTIFF
# must be given any *other* name, otherwise nothing is loaded and the function
# raises "raster_transform could not be set".
LAYER_PATHS = {
    # Topographic layers (DEM, slope)
    "dem": str(MAPPING_DIR / "DEM_east_alligator_3112.tif"),
    "slope": str(MAPPING_DIR / "slope_east_alligator_3112.tif"),

    # Categorical landcover layers (one-hot encoded)
    "landcover": str(MAPPING_DIR / "landcover_dummy_east_alligator.tif"),
}

# The training run to predict from — where its checkpoints and scaling
# coefficients were written by deepssf_piggies_train_landcover.ipynb, and where
# the outputs below are saved.
OUTPUT_DIR = Path("outputs/landcover/2026-11-17_1")
CKPT_PATH = OUTPUT_DIR / "best_model.pt"
SCALING_PATH = OUTPUT_DIR / "scalar_scaling.csv"  # weather-covariate scaling used in training

for path in (CKPT_PATH, SCALING_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"No {path.name} in {OUTPUT_DIR} — set OUTPUT_DIR to a directory that "
            "deepssf_piggies_train_landcover.ipynb has finished writing to."
        )
print(f"Output directory: {OUTPUT_DIR}")
print("Checkpoints:", ", ".join(sorted(p.name for p in OUTPUT_DIR.glob("*.pt"))))

# Import data
The csv file is imported with a timestamp and an x and y coordinate. In our case the data was already in a projected format that is specific to Australia (EPSG:3112), which is the same crs as the raster data which will be imported.

In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print(f"Raw GPS fixes : {len(raw_df):,}")
print(f"Individuals   : {raw_df['name'].nunique()}")

# Sort by the individual ID and timestamp to ensure correct temporal order
raw_df = raw_df.sort_values(by=["name", "t1_"]).reset_index(drop=True)

# Convert the 'time' column to the correct timezone (here Australia/Darwin), then
# drop the offset.  prepare_movement_df() re-parses timestamps with utc=True, so a
# tz-aware Darwin column would be converted straight back to UTC and hour_t1 would
# be 9.5 h out — making the diel covariates wrong.  Localising to Darwin and then
# going tz-naive keeps the local clock time that the sin/cos encodings need.
raw_df["t1_"] = (
        pd.to_datetime(raw_df["t1_"], utc=True)
        .dt.tz_convert("Australia/Darwin")
        .dt.tz_localize(None)
    )

raw_df.head()

## `load_environmental_layers()`

Reads each GeoTIFF listed in LAYER_PATHS, min–max

- Scales every band to [0, 1], and returns:
    * `env_layers`       — dict mapping layer name → NumPy array: 2-D (H × W) for single-band GeoTIFFs, 3-D (bands × H × W) for multi-band ones, where each band becomes its own model input channel
    * `raster_transform` — Affine object that converts pixel (row, col) indices to the projected coordinates (easting, northing) of the raster(s), which should be in the same CRS as the GPS data.  This transform is used whenever we need to translate between pixel space and real-world coordinates.

**Two things to watch for:**

1. The keys `'s2'` and `'s2_dir'` are *reserved* for the monthly Sentinel-2 composite loader — a single GeoTIFF passed under the name `'s2'` is silently skipped (see the note in the paths cell above).
2. The rescaling uses a **single global min/max across all bands** of a file, and computes `(data - lo) / hi` rather than `(data - lo) / (hi - lo)`. Our index raster is `int16` scaled by 10 000 with ±10 000 fill values, so the loaded values span [0, 2] rather than [0, 1] and the three indices are not individually normalised. That is workable (the CNN can rescale internally) but if it hurts training, pre-scale each band yourself and save as `.npy` — non-`.tif` paths in `LAYER_PATHS` are loaded with `np.load(..., mmap_mode='r')` and left untouched, though at least one real `.tif` must remain in the dict so the transform can be set.


In [ ]:
env_layers, raster_transform = deepssf.load_environmental_layers(LAYER_PATHS)
print("Layers loaded:", list(env_layers.keys()))
for name, arr in env_layers.items():
    print(f"  {name}: shape={arr.shape}, min={arr.min():.3f}, max={arr.max():.3f}")

# Each band of a multi-band layer becomes one CNN input channel, so count bands
# (not layers) here and reuse the total when setting ModelParams below.  Deriving
# it means switching rasters in LAYER_PATHS needs no other edits.
N_SPATIAL_CHANNELS = sum(1 if arr.ndim == 2 else arr.shape[0] for arr in env_layers.values())
print(f"Spatial channels: {N_SPATIAL_CHANNELS}")

## `SCALAR_COLS`
These are non-spatial inputs. For processing with convolutional layers, they are converted into spatial windows (rasters/arrays), and stacked with the truly spatial covariates.

The model receives a fixed-size spatial window centred on each step's start location.  To let the network express time-varying habitat preferences (e.g. different nocturnal vs diurnal use), scalar covariates are tiled into additional channels of the same H × W size, so every pixel "knows" the current time context when habitat suitability is estimated.

- `hour_t1_sin1` / `hour_t1_cos1`
    * Sine and cosine of (2π × hour / 24), encoding time-of-day as a *circular* variable.  The circular encoding means 23:00 and 01:00 are treated as adjacent — a plain hour value would have an artificial discontinuity at midnight.

- `yday_t1_sin1` / `yday_t1_cos1`
    * Same circular encoding for day-of-year (1–365), capturing seasonal variation in habitat use (e.g. dry-season vs wet-season movements).

- `dt_hour`
    * Elapsed time between consecutive fixes (hours).  Because GPS collars do not always fix at a perfectly regular rate, including dt_hour lets the movement kernel rescale expected displacement proportionally to the actual time interval rather than assuming a fixed fix rate. Note: dt_hour must be supplied at *simulation* time as well, so that `simulate_trajectory()` produces inputs consistent with training.

- Weather covariates from the `onpli_` weather station, carried over from the CSV for the start of each step:
    * `rain_30min_mm`, `air_temp_c`, `wet_bulb_temp_c`, `dew_point_temp_c`, `rel_humidity_pct`, `wind_speed_kmh`
        - Standardised to z-scores (the `*_scaled` columns) before going into the model, with rain log-transformed, log(1 + x), first.  The mean and SD are kept so model inputs and effects can be converted back to natural units.
    * `wind_dir_deg`
        - Encoded as a sine/cosine pair (`onpli_wind_dir_sin1` / `onpli_wind_dir_cos1`) for the same wraparound reason as hour of day, with calm steps set to (0, 0).

These must match `SCALAR_COLS` in the training notebook — same columns, same order.

In [ ]:
# Time covariates derived by prepare_movement_df().  Not scaled: the sin/cos pairs
# are already in [-1, 1], and simulate_trajectory() fills dt_hour with the raw
# time_between_steps (hours), so the model has to be trained on raw hours too.
TIME_COLS = ["hour_t1_sin1",
             "hour_t1_cos1",
             "yday_t1_sin1",
             "yday_t1_cos1",
             "dt_hour"]

# Weather covariates carried over from the CSV in natural units, and standardised
# to z-scores further down (the "_scaled" columns)
WEATHER_COLS = ["onpli_rain_30min_mm",
                "onpli_air_temp_c",
                "onpli_wet_bulb_temp_c",
                "onpli_dew_point_temp_c",
                "onpli_rel_humidity_pct",
                "onpli_wind_speed_kmh"
                ]

# Wind direction (degrees) is carried over, then encoded as a sin/cos pair further
# down rather than scaled
WIND_DIR_COL = "onpli_wind_dir_deg"

# The model inputs, in the order the model sees them
SCALAR_COLS = (
    TIME_COLS
    + [f"{col}_scaled" for col in WEATHER_COLS]
    + ["onpli_wind_dir_sin1", "onpli_wind_dir_cos1"]
)
print(f"{len(SCALAR_COLS)} scalar inputs: {SCALAR_COLS}")

## `prepare_movement_df()`
Converts a fixes dataframe (one row per GPS location) into a *step* dataframe (one row per movement between consecutive fixes).

- For each step it computes:
    * `dx`, `dy`         — displacement in projected CRS units (metres)
    * `bearing`          — direction of travel (radians; 0 = north, clockwise positive)
    * `bearing_tm1`      — bearing of the *previous* step, used as a model input so the network can capture directional persistence by estimating a turning angle
    * `dt_hour`          — time elapsed between consecutive fixes (hours)
    * `hour_t1`, `yday_t1` — decimal hour-of-day and day-of-year at the step start location
    * `*_sin1` / `*_cos1`  — sine/cosine encodings of hour and yday (see SCALAR_COLS below)

The "1" on column names (x1_, y1_, t1_, etc.) flags them as "start of step" variables; and the "2" in x2_, y2_, t2_ are the corresponding "end of step" values, following the convention of the `amt` package [@Signer2019-fi]. The last fix for each individual is dropped because no forward step can be formed.

In [ ]:
# The CSV is already in amt step format, but prepare_movement_df() expects one row
# per *fix* and rebuilds the steps itself (it also derives bearing, dt_hour and the
# cyclic time encodings the model needs).  Point it at the step-start columns and
# let it re-form the steps from consecutive fixes.
step_df = deepssf.prepare_movement_df(
    raw_df, 
    id_col="name", 
    time_col="t1_", 
    x_col="x1_", 
    y_col="y1_",
    # derives the time cols; carries the weather and wind direction cols over
    scalar_cols=TIME_COLS + WEATHER_COLS + [WIND_DIR_COL],
)
n_dropped = len(raw_df) - len(step_df)
print(f"Movement steps : {len(step_df):,}  ({n_dropped} dropped — last per individual)")
step_df.head()

### Check and drop NAs

In [ ]:
# Columns that must be complete for a step to be used
NA_CHECK_COLS = TIME_COLS + WEATHER_COLS + [WIND_DIR_COL]

# --- Check: missing values per column ---
na_mask = step_df[NA_CHECK_COLS].isna()
print("Missing values per column:")
print(na_mask.sum().to_string())

# --- Drop: every step with a missing value in any of these columns ---
drop_rows = na_mask.any(axis=1)
if drop_rows.any():
    # Record which column(s) caused each dropped step (kept for inspection)
    dropped_steps = step_df.loc[drop_rows, ["id", "t1_"]].copy()
    dropped_steps["na_cols"] = na_mask[drop_rows].apply(
        lambda row: ", ".join(row.index[row]), axis=1
    )
    print(f"\nDropping {drop_rows.sum():,} of {len(step_df):,} steps "
          f"({drop_rows.mean():.2%}).  Column(s) with NAs that caused the drop:")
    print(dropped_steps["na_cols"].value_counts().to_string())
    step_df = step_df.loc[~drop_rows].reset_index(drop=True)
else:
    print("\nNo steps with missing values.")

print(f"\nSteps remaining: {len(step_df):,}")

### Encode wind direction as a sin/cos pair
Wind direction follows the BoM convention: the compass bearing the wind blows *from*, in 10° steps clockwise from north (90 = from the east).  North is recorded as 360, and 0 is reserved for **calm** (no wind, so no direction).

Taking the sine and cosine removes the wraparound, so 350° and 10° sit next to each other and 0° and 360° both mean north:
- `onpli_wind_dir_sin1` — east–west component (+1 = from the east, −1 = from the west)
- `onpli_wind_dir_cos1` — north–south component (+1 = from the north, −1 = from the south)

Calm steps are set to (0, 0).  A plain sin/cos of 0° gives (0, 1), which the model would read as a northerly.  `wind_dir_from_sin_cos()` converts back to 0–360° (0 = north), returning NaN for calm.

In [ ]:
def wind_dir_to_sin_cos(deg):
    """Compass degrees (0 = north, clockwise) -> (sin, cos)."""
    rad = np.radians(np.asarray(deg, dtype=float))
    return np.sin(rad), np.cos(rad)


def wind_dir_from_sin_cos(sin1, cos1):
    """(sin, cos) -> compass degrees in [0, 360), 0 = north.  NaN for calm (0, 0)."""
    sin1 = np.asarray(sin1, dtype=float)
    cos1 = np.asarray(cos1, dtype=float)
    # Round before wrapping so north comes back as 0 rather than 359.99999...
    deg = np.round(np.degrees(np.arctan2(sin1, cos1)), 6) % 360
    return np.where(np.hypot(sin1, cos1) < 1e-6, np.nan, deg)


calm = (step_df[WIND_DIR_COL] == 0).to_numpy()   # BoM code for calm
wind_sin, wind_cos = wind_dir_to_sin_cos(step_df[WIND_DIR_COL])
step_df["onpli_wind_dir_sin1"] = np.where(calm, 0.0, wind_sin)
step_df["onpli_wind_dir_cos1"] = np.where(calm, 0.0, wind_cos)

# --- Checks ---
# 1. Calm (direction 0) should coincide with zero wind speed
print(f"Calm steps (direction 0): {calm.sum():,}, of which wind speed is 0: "
      f"{(step_df.loc[calm, 'onpli_wind_speed_kmh'] == 0).sum():,}\n")

# 2. Compass points encode and decode as expected, including across 360/0
check = pd.DataFrame({"deg": [360, 10, 90, 180, 270, 350]})
check["sin1"], check["cos1"] = wind_dir_to_sin_cos(check["deg"])
check["recovered_deg"] = wind_dir_from_sin_cos(check["sin1"], check["cos1"])
print(check.round(3).to_string(index=False))

# 3. Round trip on the data recovers every non-calm direction (360 comes back as 0)
#    and returns NaN for every calm step
recovered = wind_dir_from_sin_cos(step_df["onpli_wind_dir_sin1"], step_df["onpli_wind_dir_cos1"])
diff = (recovered[~calm] - step_df.loc[~calm, WIND_DIR_COL].to_numpy() + 180) % 360 - 180
print(f"\nMax round-trip error (non-calm): {np.abs(diff).max():.1e} deg;  "
      f"calm steps recovered as NaN: {np.isnan(recovered[calm]).all()}")

### Scale the weather covariates with the training coefficients
The model was trained on weather covariates scaled with a particular transform, mean and SD (rain log-transformed, log(1 + x), then every covariate converted to a z-score).  Any input to the model has to be scaled the same way, so the coefficients are loaded from `scalar_scaling.csv`, which the training notebook saved in `OUTPUT_DIR`, rather than recalculated here.

- `scale_covariate(values, col)` converts natural units to the model's scale, e.g. for weather values set by hand for a prediction.
- `unscale_covariate(values, col)` converts back to natural units: natural = scaled × SD + mean, followed by exp(x) − 1 for log1p columns.

As a check, the coefficients are also recalculated from the data loaded here.  They should match, unless the data or the NA handling has changed since the model was trained.

In [ ]:
# Transform, mean and SD for each weather covariate, as used in training
SCALING = pd.read_csv(SCALING_PATH, index_col="column")
missing = [col for col in WEATHER_COLS if col not in SCALING.index]
if missing:
    raise KeyError(f"{SCALING_PATH} has no scaling for {missing}")
print("Scaling coefficients (from training):")
print(SCALING.round(3).to_string())


def scale_covariate(values, col):
    """Natural units -> model scale for `col`: log1p if listed, then z-score."""
    values = np.asarray(values, dtype=float)
    if SCALING.at[col, "transform"] == "log1p":
        values = np.log1p(values)
    return (values - SCALING.at[col, "mean"]) / SCALING.at[col, "sd"]


def unscale_covariate(values, col):
    """Model scale -> natural units for `col`: undo the z-score, then expm1 if logged."""
    values = np.asarray(values, dtype=float) * SCALING.at[col, "sd"] + SCALING.at[col, "mean"]
    if SCALING.at[col, "transform"] == "log1p":
        values = np.expm1(values)
    return values


for col in WEATHER_COLS:
    step_df[f"{col}_scaled"] = scale_covariate(step_df[col], col)

# Check: recalculating the coefficients from this notebook's data gives the same
# values as the training run
for col in WEATHER_COLS:
    values = step_df[col].to_numpy(dtype=float)
    if SCALING.at[col, "transform"] == "log1p":
        values = np.log1p(values)
    if not np.allclose([values.mean(), values.std(ddof=1)],
                       SCALING.loc[col, ["mean", "sd"]].to_numpy(dtype=float)):
        print(f"Warning: {col} mean/SD differ from training — has the data or NA "
              "handling changed since the model was trained?")
        break
else:
    print("\nCoefficients recalculated from this data match the training run.")

## `WINDOW_SIZE`
The size of the local availability window in cells/pixels - should match what was in the training script. 

In [ ]:
WINDOW_SIZE = 51  # the window the model in OUTPUT_DIR was trained with

## `PIXEL_SIZE`
Raster resolution in CRS units (metres here).  Read straight off `raster_transform`
rather than typed in: the transform is the authority on the grid the model is
trained against, and a hand-entered value that disagrees with it silently
mis-scales the movement kernel — which is parameterised in metres — against a
grid measured in pixels.

In [ ]:
# transform.a is the cell width, transform.e the (negative) cell height
PIXEL_SIZE = abs(raster_transform.a)

if not math.isclose(abs(raster_transform.a), abs(raster_transform.e)):
    raise ValueError(
        f"Non-square pixels ({abs(raster_transform.a)} x {abs(raster_transform.e)}); "
        "the model assumes square cells."
    )
print(f"Pixel size: {PIXEL_SIZE} m  →  window {WINDOW_SIZE} px = "
      f"{WINDOW_SIZE * PIXEL_SIZE:,.0f} m across")

## `filter_steps_by_window()` 
Removes steps whose straight-line displacement exceeds half the spatial window (WINDOW_SIZE × PIXEL_SIZE / 2 metres). Such steps would place the true next location outside the predicted probability map, causing an index error in the loss function.

In [ ]:
step_df = deepssf.filter_steps_by_window(
    step_df, window_size=WINDOW_SIZE, pixel_size=PIXEL_SIZE
)
print(f"Steps after window filter: {len(step_df):,}")

## `params_from_checkpoint()`
Weights only load into a model built with the *same* architecture, so predicting
from a trained model means reproducing its hyper-parameters exactly.  Rather
than re-typing the training notebook's constants here — where a habitat branch
one layer too shallow fails loudly, but a mis-set `pixel_size` quietly rescales
the movement kernel — `params_from_checkpoint()` recovers them from the
checkpoint and returns the `ModelParams` to hand to `ConvJointModel`.

`deepssf` writes the `ModelParams` dict into every checkpoint, so for a model
trained with 0.3.1 or later it is simply read back.  Older checkpoints (anything
in `outputs/` trained before then) do not have it, and the architecture is
recovered from the layer shapes instead: a conv weight is
`[out_channels, in_channels, k, k]` and a linear weight
`[out_features, in_features]`, which between them pin down every hyper-parameter
that changes a tensor shape — `input_channels`, `output_channels`,
`kernel_size`, both conv-layer counts, `dense_dim_in_all`, `dense_dim_hidden`
and `num_movement_params`.  Either way the result is checked against the saved
tensors, so a stored params dict that contradicts its own weights raises here
rather than failing later inside `load_state_dict`.

Three things it cannot get from the file, passed in below:

* `image_dim` — the habitat branch is fully convolutional, so nothing in the
  weights records the training window.  The movement branch's flattened size
  does constrain it, and that is checked, but only to within the slack that
  max-pooling's floor division allows (48–55 px all flatten to the same 288
  here).  `WINDOW_SIZE` also has to match `filter_steps_by_window()` above, and
  is the window the simulations step within.
* `pixel_size` — a property of the raster, read off `raster_transform`.
* `device` — where the model is being *run*, which has nothing to do with where
  it was trained; it defaults to this machine rather than to the recorded value.

`n_scalar_covariates` only fills the two `ModelParams` fields that no layer
reads; the scalar count that matters is folded into `input_channels`, which
comes off the weights.  The check below is the one thing the checkpoint cannot
do for us — that the covariates loaded in *this* notebook are the ones the model
was trained on.  It compares counts only, so keep `LAYER_PATHS` in the training
run's order.


In [ ]:
params = deepssf.params_from_checkpoint(
    str(CKPT_PATH),
    image_dim=WINDOW_SIZE,               # not recorded in the weights
    pixel_size=PIXEL_SIZE,               # from raster_transform
    n_scalar_covariates=len(SCALAR_COLS),
    device=DEVICE,                       # this machine, not the training machine
)
for name, value in params.to_dict().items():
    print(f"  {name:26s}: {value}")

# The covariates in this notebook must be the ones the model was trained on.
# input_channels is one channel per raster band plus one per scalar covariate,
# so a changed LAYER_PATHS or SCALAR_COLS shows up here as a count mismatch.
n_expected = N_SPATIAL_CHANNELS + len(SCALAR_COLS)
if n_expected != params.input_channels:
    raise ValueError(
        f"{N_SPATIAL_CHANNELS} raster bands + {len(SCALAR_COLS)} scalars = {n_expected} "
        f"input channels, but the checkpoint was trained with {params.input_channels}. "
        "LAYER_PATHS or SCALAR_COLS differ from the training run."
    )
print(f"\n{N_SPATIAL_CHANNELS} raster bands + {len(SCALAR_COLS)} scalars = "
      f"{params.input_channels} input channels - matches the checkpoint")


## `ConvJointModel` 
The deepSSF model! A joint habitat–movement model with two heads:
1. Habitat head 
    — a small CNN applied to the spatial window (env layers + tiled scalars) that outputs a per-pixel log-suitability map.
2. Movement head 
    — a dense MLP (fully-connected layer) that takes the scalar covariates and the previous bearing, and produces parameters for a parametric movement kernel (a mixture of wrapped distributions centred on the current location).

The two log-probability surfaces are summed pixel-wise to give the joint predicted probability map for the animal's next step.

In [ ]:
model = deepssf.ConvJointModel(params).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")

## Load the trained model
`ConvJointModel` above is freshly initialised — random weights — so nothing it
predicts means anything until a checkpoint is loaded into it.  A training run
writes three:

- `best_model.pt` — the model to use for **simulation**.  Both heads from one
  epoch, so the joint likelihood is the one that was actually measured.
- `best_model_habitat.pt` / `best_model_movement.pt` — each head at its own best
  epoch, tracked across the whole run.

Mixing them is a deliberate trade: taking the habitat head from a different
epoch gives the sharpest habitat surface to *map*, but the two heads then come
from different epochs and the joint model's likelihood is no longer what either
checkpoint recorded.  Use it for the landscape predictions below, not for
simulating trajectories.

`load_checkpoint()` loads the weights in place and is strict about shapes, so a
mismatch between the constants set above and the trained architecture raises
here rather than silently predicting from a partly-random model.


In [ ]:
# The joint model — what the simulations further down should use
ckpt = deepssf.load_checkpoint(str(CKPT_PATH), model)
print(f"best_model.pt          saved on '{ckpt['checkpoint_criterion']}' "
      f"= {ckpt['checkpoint_score']:.6f}")
print(f"                       trained with deepssf {ckpt.get('deepssf_version', 'unknown')}, "
      f"val_loss {ckpt['val_loss']:.6f}")

hab_ckpt = deepssf.load_checkpoint(str(OUTPUT_DIR / "best_model_habitat.pt"))
print(f"best_model_habitat.pt  saved on '{hab_ckpt['checkpoint_criterion']}' "
      f"= {hab_ckpt['checkpoint_score']:.6f}")
print(f"(a uniform surface over the window scores {np.log(WINDOW_SIZE ** 2):.4f})")

# Prediction only — no gradients needed, and this turns off dropout/batchnorm
# training behaviour.
model.eval()

# To map the sharpest habitat surface instead, overwrite just that head:
# load_head_weights(str(OUTPUT_DIR / "best_model_habitat.pt"), model, "habitat")


## Convert the NumPy rasters loaded earlier into PyTorch tensors
`predict_habitat_landscape()`, `simulate_trajectory()` and
`simulate_trajectories()` all call `get_landscape(month_index)` to fetch the
current environmental layers as a list of 2-D tensors.

With static (non-seasonal) rasters the `month_index` argument is unused. For
seasonal data (e.g. monthly NDVI composites) you would index into a stack of
layers here based on the current month.


In [ ]:
# The simulation and prediction functions stack the list returned by
# get_landscape() along a new channel dimension, so every element must be a *2-D*
# tensor.  Unbind the band dimension of any multi-band layer so the channel order
# matches what MovementDataset fed the model during training.
landscape_tensors = []
for arr in env_layers.values():
    t = torch.from_numpy(np.asarray(arr).astype(np.float32))
    landscape_tensors.extend([t] if t.ndim == 2 else list(t))

print(f"Landscape channels: {len(landscape_tensors)}  (each {tuple(landscape_tensors[0].shape)})")


def get_landscape(_month_index):
    # Returns one 2-D tensor per spatial channel, in the same order as LAYER_PATHS.
    return landscape_tensors


# Habitat selection across the full landscape

## Scalar inputs for the prediction
The model needs a value for every column in `SCALAR_COLS`, in that order:
- **Time** (`hour_t1_*`, `yday_t1_*`, `dt_hour`)
    - `deepssf.make_simulation_inputs()` builds the sin/cos encodings from a day of the year (`PRED_YDAY`, integer) and a fractional hour (`PRED_HOUR`, float).  Its `time_between_steps` argument becomes the `dt_hour` input, so it is set to the collars' ~30 min fix interval (`PRED_DT_HOUR`) rather than left at its default of 1 hour.
- **Weather** (`*_scaled`)
    - set by hand in natural units in `PRED_WEATHER`, then scaled with the coefficients stored in `SCALING`.
- **Wind direction** (`onpli_wind_dir_sin1` / `_cos1`)
    - set in degrees in `PRED_WIND_DIR_DEG` (0 or 360 = north, the direction the wind blows *from*), and encoded as a sin/cos pair, or (0, 0) when the wind speed is 0 (calm).

Weather varies a lot with season and time of day, so the first cell summarises the training data, both overall and for steps near the chosen moment, as a guide to realistic values.  `make_scalar_row()` then assembles the full row, warning about any weather value outside the training range.

In [ ]:
# Predict for one moment in time: 9:30 am on day 200.
PRED_YDAY = 200  # day of the year to predict for (1–365, integer)
PRED_HOUR = 9.5  # hour of the day to predict for (0–24, float)

# Steps "near" the chosen moment: within ±15 days and ±1 hour, wrapping across
# the new year and midnight
NEAR_DAYS, NEAR_HOURS = 15, 1
yday_diff = (step_df["yday_t1"] - PRED_YDAY + 182.5) % 365 - 182.5
hour_diff = (step_df["hour_t1"] - PRED_HOUR + 12) % 24 - 12
near = (yday_diff.abs() <= NEAR_DAYS) & (hour_diff.abs() <= NEAR_HOURS)

print("Model inputs needed (SCALAR_COLS):")
for col in SCALAR_COLS:
    if col in TIME_COLS:
        source = "PRED_YDAY / PRED_HOUR / PRED_DT_HOUR"
    elif col.endswith("_scaled"):
        source = f'PRED_WEATHER["{col.removesuffix("_scaled")}"]'
    else:
        source = "PRED_WIND_DIR_DEG"
    print(f"  {col:<32} <- {source}")

# Weather in natural units: the full training range, and typical values near the
# chosen moment
near_weather = step_df.loc[near, WEATHER_COLS]
weather_summary = pd.DataFrame({
    "min":        step_df[WEATHER_COLS].min(),
    "mean":       step_df[WEATHER_COLS].mean(),
    "max":        step_df[WEATHER_COLS].max(),
    "near_5%":    near_weather.quantile(0.05),
    "near_median": near_weather.median(),
    "near_95%":   near_weather.quantile(0.95),
})
print(f"\nWeather, all steps vs the {near.sum():,} steps within ±{NEAR_DAYS} days "
      f"and ±{NEAR_HOURS} h of day {PRED_YDAY}, {PRED_HOUR:g} h:")
print(weather_summary.round(2).to_string())

# Wind direction is circular, so a mean is meaningless: show the most common
# directions instead (0 = calm is counted separately)
near_dir = step_df.loc[near, WIND_DIR_COL]
print(f"\nWind direction near the chosen moment: {(near_dir == 0).mean():.0%} calm; "
      "most common directions (deg from north, % of non-calm steps):")
print((near_dir[near_dir > 0].value_counts(normalize=True).head(5) * 100).round(1).to_string())

In [ ]:
# Weather for the prediction, in natural units.  The defaults are roughly the
# medians near day 200 at 09:30 (dry-season mornings).  Temperature, wet bulb,
# dew point and humidity are physically linked, so change them together, e.g. by
# taking them from the same row of the summary above.
PRED_WEATHER = {
    "onpli_rain_30min_mm":    0.0,
    "onpli_air_temp_c":       27.0,
    "onpli_wet_bulb_temp_c":  19.8,
    "onpli_dew_point_temp_c": 15.0,
    "onpli_rel_humidity_pct": 47.0,
    "onpli_wind_speed_kmh":   15.0,
}

# Direction the wind blows FROM, in degrees clockwise from north (0 or 360 =
# north).  Ignored when the wind speed is 0, which is encoded as calm.
PRED_WIND_DIR_DEG = 110.0

# Hours between steps, which is also the model's dt_hour input.  The collars fixed
# about every 30 min, so values far from 0.5 are outside what the model saw.
PRED_DT_HOUR = 0.5


def make_scalar_row(yday, hour, weather=None, wind_dir_deg=None, dt_hour=None):
    """One row of model inputs, in SCALAR_COLS order, from natural-unit values.

    Defaults to PRED_WEATHER, PRED_WIND_DIR_DEG and PRED_DT_HOUR.  Prints a
    warning for any weather value outside the range seen in training.
    """
    weather = PRED_WEATHER if weather is None else weather
    wind_dir_deg = PRED_WIND_DIR_DEG if wind_dir_deg is None else wind_dir_deg
    dt_hour = PRED_DT_HOUR if dt_hour is None else dt_hour

    missing = [col for col in WEATHER_COLS if col not in weather]
    if missing:
        raise KeyError(f"weather is missing {missing}")

    # Time: same encodings the simulation uses, in TIME_COLS order
    time_row = deepssf.make_simulation_inputs(
        n_steps=1, starting_yday=yday, starting_hour=hour, time_between_steps=dt_hour,
    )[0][0]
    values = dict(zip(TIME_COLS, time_row, strict=True))

    # Weather: scaled with the training coefficients
    for col in WEATHER_COLS:
        lo, hi = step_df[col].min(), step_df[col].max()
        if not lo <= weather[col] <= hi:
            print(f"Warning: {col} = {weather[col]} is outside the training range ({lo} to {hi})")
        values[f"{col}_scaled"] = scale_covariate([weather[col]], col)[0]

    # Wind direction: sin/cos, or (0, 0) for calm
    if weather["onpli_wind_speed_kmh"] == 0:
        wind_sin, wind_cos = 0.0, 0.0
    else:
        wind_sin, wind_cos = wind_dir_to_sin_cos(wind_dir_deg)
    values["onpli_wind_dir_sin1"] = wind_sin
    values["onpli_wind_dir_cos1"] = wind_cos

    return np.array([values[col] for col in SCALAR_COLS], dtype=np.float32)


# Check: rebuilding rows of the training data from their natural-unit values gives
# exactly the model inputs the model was trained on
check_rows = step_df[step_df["yday_t1"] <= 365].sample(200, random_state=1)
for _, row in check_rows.iterrows():
    rebuilt = make_scalar_row(
        row["yday_t1"], row["hour_t1"],
        weather=row[WEATHER_COLS].to_dict(),
        wind_dir_deg=row[WIND_DIR_COL],
        dt_hour=row["dt_hour"],
    )
    assert np.allclose(rebuilt, row[SCALAR_COLS].to_numpy(dtype=float), atol=1e-5), row.name
print(f"make_scalar_row() reproduces the training inputs for {len(check_rows)} sampled steps\n")

scalars_pred = make_scalar_row(PRED_YDAY, PRED_HOUR)
print(f"Model inputs for day {PRED_YDAY}, {PRED_HOUR:g} h:")
print(pd.Series(scalars_pred, index=SCALAR_COLS).round(3).to_string())

## `predict_habitat_landscape()`
Everything so far has run the model on a `WINDOW_SIZE` crop centred on the
animal.  But the habitat sub-network is *fully convolutional* — 3x3
convolutions with stride 1 and padding 1, no pooling and no dense layer — so the
same trained filters can be swept across the entire raster in a single pass.
Convolution is translation-equivariant, so the value at a pixel is exactly what
the model would have produced for a window centred on that pixel.

That gives a landscape-wide habitat selection surface.  Read it as a **relative
ranking of cells**: "if the animal were somewhere on this landscape, how
attractive is each cell". It deliberately says nothing about whether the animal
could *reach* a cell in one step — that constraint lives in the movement kernel,
which takes no part here.  The two are only combined over a local window, which
is what `simulate_next_step()` does.

- **Inputs**:
    * `model`
        - the trained model; only `model.conv_habitat` is used
    * `landscape_rasters`
        - the spatial covariates as a list of 2-D tensors, in the same order and
          on the same scaling as training — i.e. exactly what `get_landscape()`
          returns
    * `scalars`
        - the scalar covariates for the moment being predicted, in `SCALAR_COLS`
          order, as built by `make_scalar_row()` above.  Each value is
          broadcast to a constant layer covering the whole landscape (the same
          thing `Scalar_to_Grid_Block` does for a window — that block can't be
          reused here because its output size is fixed to `WINDOW_SIZE`)
    * `normalise`
        - subtract the log-sum-exp over the whole landscape, making the surface
          a log-probability that sums to 1 across all valid cells
    * `edge_buffer`
        - how many border pixels to mask as `NaN`.  Defaults to
          `habitat_edge_buffer(model)` — the exact width the zero-padding
          reaches, 1 pixel per habitat conv layer.  Those pixels saw padding
          rather than landscape and are not comparable with the interior
    * `chunk_rows`
        - process this many raster rows at a time.  A full-landscape activation
          is several hundred MB, enough to exhaust a GPU; the chunks overlap by
          `edge_buffer` so the result is identical to a single pass

- **Outputs**: a `[height, width]` NumPy array of log-probabilities, `NaN` in the
  border.  Take `np.exp()` for probabilities.

In [ ]:
habitat_log = deepssf.predict_habitat_landscape(
    model,
    landscape_rasters=get_landscape(0),   # static layers → month index 0
    scalars=scalars_pred,
)
print(f"\nSurface {habitat_log.shape}, masked border "
      f"{deepssf.habitat_edge_buffer(model)} px")
print(f"log-probability range: {np.nanmin(habitat_log):.2f} to "
      f"{np.nanmax(habitat_log):.2f}")
print(f"probabilities sum to  : {np.nansum(np.exp(habitat_log)):.4f}")

### Plot the landscape-scale habitat selection surface
Plotted on the **log** scale.  Across 9.5 million cells the probabilities are
tiny (a uniform surface would give ~1e-7 per cell) and a handful of very
attractive cells dominate the linear range, so a linear colour scale washes the
whole map out.  The colour limits are clipped to percentiles for the same
reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Percentile limits: a few extreme cells otherwise dominate the colour range
vmin, vmax = np.nanpercentile(habitat_log, [1, 99.9])

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[0],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[0].set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, "
                  f"{PRED_HOUR:.0f}:00")

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[1],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[1].plot(
    step_df["x1_"], step_df["y1_"],
    "r.", alpha=0.25, markersize=0.6, label="observed GPS fixes",
)
axes[1].legend(loc="upper right", markerscale=10)
axes[1].set_title("with observed locations overlaid")

for ax in axes:
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")

# One colourbar for both panels
fig.colorbar(axes[0].images[0], ax=axes, shrink=0.8, label="log probability")
plt.savefig(str(OUTPUT_DIR / "habitat_landscape.png"), dpi=120, bbox_inches="tight")
plt.show()